# FDS Challenge: My Notebook

## 0. Configuration

In [ ]:
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split


DEBUG = False

pd.set_option("display.max_columns", None)  # keep it! it's usefull to make pandas print also the null columns
pd.set_option("display.width", None)


# --- PATHS ---
TRAIN_FILE_PATH        = "../dataset/train.jsonl"
TEST_FILE_PATH         = "../dataset/test.jsonl"
TYPE_CHART_PATH        = "../dataset/type_chart_gen2.csv"
SAMPLE_SUBMISSION_PATH = "../dataset/sample_submission.csv"
SUBMISSION_PATH        = "submission.csv"
REPORT_DIR             = "reports"  # One timestamped Markdown report per run.

Path("../temp").mkdir(exist_ok=True)


# --- FEATURES ---
TEAM_SIZE = 6

STATISTICS = ["hp", "atk", "def", "spa", "spd", "spe"]
HP_PCT_LOST_INTERVALS = ["0_10", "10_20", "20_30"]
MULTIPLIERS = [0, 0.25, 0.5, 1, 2, 4]

STATUSES_WEIGHTS = {
    "fnt": 6,
    "slp": 2,
    "frz": 4,
    "tox": 3,
    "psn": 1,
    "par": 2,
    "brn": 2
}

BOOSTS_WEIGHTS = {
    "atk": 2,
    "def": 1,
    "spa": 2,
    "spd": 1,
    "spe": 2
}

MULTS_WEIGHTS = {
    0:      -4,
    0.25: -1.5,
    0.5:  -0.5,
    1:       0,
    2:     1.5,
    4:       3
}

EFFECT_WEIGHTS = {
    "attract":      -0.5,
    "beatup":        0.3,
    "bide":          0.4,
    "bind":         -0.4,
    "clamp":        -1.0,
    "confusion":    -0.8,
    "curse":        -0.8,
    "destinybond":   0.6,
    "disable":      -0.4,
    "encore":       -1.2,
    "endure":       -0.5,
    "firespin":     -0.4,
    "focusband":    -0.8,
    "focusenergy":  -0.4,
    "foresight":    -0.2,
    "futuresight":  -0.7,
    "leechseed":    -0.8,
    "lockon":        0.2,
    "magnitude":     0.3,
    "mimic":        -0.6,
    "mindreader":   -0.5,
    "mysteryberry": -0.3,
    "nightmare":    -1.0,
    "noeffect":      0.0,
    "perish":        0.0,
    "protect":       0.0,
    "pursuit":      -1.5,
    "quickclaw":    -0.6,
    "safeguard":     0.6,
    "spite":        -0.3,
    "substitute":    1.0,
    "trapped":      -0.9,
    "typechange":    0.3,
    "whirlpool":    -0.4,
    "wrap":         -0.4
}

ALL_FEATURES = [
    # BASE STATS

    "p1_sum_hp",
    "p1_sum_atk",
    "p1_sum_def",
    "p1_sum_spa",
    "p1_sum_spd",
    "p1_sum_spe",
    "p1_mean_hp",
    "p1_mean_atk",
    "p1_mean_def",
    "p1_mean_spa",
    "p1_mean_spd",
    "p1_mean_spe",
    "p1_min_base_hp",
    "p1_min_base_atk",
    "p1_min_base_def",
    "p1_min_base_spa",
    "p1_min_base_spd",
    "p1_min_base_spe",
    "p1_max_base_hp",
    "p1_max_base_atk",
    "p1_max_base_def",
    "p1_max_base_spa",
    "p1_max_base_spd",
    "p1_max_base_spe",
    "p1_std_base_hp",
    "p1_std_base_atk",
    "p1_std_base_def",
    "p1_std_base_spa",
    "p1_std_base_spd",
    "p1_std_base_spe",
    "p2_sum_hp",
    "p2_sum_atk",
    "p2_sum_def",
    "p2_sum_spa",
    "p2_sum_spd",
    "p2_sum_spe",
    "p2_mean_hp",
    "p2_mean_atk",
    "p2_mean_def",
    "p2_mean_spa",
    "p2_mean_spd",
    "p2_mean_spe",
    "p2_min_base_hp",
    "p2_min_base_atk",
    "p2_min_base_def",
    "p2_min_base_spa",
    "p2_min_base_spd",
    "p2_min_base_spe",
    "p2_max_base_hp",
    "p2_max_base_atk",
    "p2_max_base_def",
    "p2_max_base_spa",
    "p2_max_base_spd",
    "p2_max_base_spe",
    "p2_std_base_hp",
    "p2_std_base_atk",
    "p2_std_base_def",
    "p2_std_base_spa",
    "p2_std_base_spd",
    "p2_std_base_spe",
    "p1_sum_bst",
    "p1_mean_bst",
    "p1_std_base_bst",
    "p2_sum_bst",
    "p2_mean_bst",
    "p2_std_base_bst",
    "diff_sum_hp",
    "diff_sum_atk",
    "diff_sum_def",
    "diff_sum_spa",
    "diff_sum_spd",
    "diff_sum_spe",
    "diff_mean_hp",
    "diff_mean_atk",
    "diff_mean_def",
    "diff_mean_spa",
    "diff_mean_spd",
    "diff_mean_spe",
    "diff_sum_bst",
    "diff_mean_bst",
    "diff_std_base_bst",

    # HP PCT LOST

    "p1_hp_pct_lost_0_10",
    "p1_hp_pct_lost_10_20",
    "p1_hp_pct_lost_20_30",
    "p1_sum_hp_lost",
    "p1_mean_hp_pct_lost",
    "p1_std_hp_pct_lost",
    "p2_hp_pct_lost_0_10",
    "p2_hp_pct_lost_10_20",
    "p2_hp_pct_lost_20_30",
    "p2_sum_hp_lost",
    "p2_mean_hp_pct_lost",
    "p2_std_hp_pct_lost",
    "diff_hp_pct_lost_0_10",
    "diff_hp_pct_lost_10_20",
    "diff_hp_pct_lost_20_30",
    "diff_sum_hp_pct_lost",
    "diff_mean_hp_pct_lost",
    "diff_std_hp_pct_lost",

    # MULTIPLIERS

    "p1_0_mult",
    "p1_0.25_mult",
    "p1_0.5_mult", 
    "p1_1_mult",
    "p1_2_mult",
    "p1_4_mult",
    "p2_0_mult",
    "p2_0.25_mult",
    "p2_0.5_mult", 
    "p2_1_mult",
    "p2_2_mult",
    "p2_4_mult",
    "diff_0_mult",
    "diff_0.25_mult",
    "diff_0.5_mult",
    "diff_1_mult",
    "diff_2_mult",
    "diff_4_mult",
    "p1_count_mult",
    "p2_count_mult",
    "p1_sum_mult",
    "p2_sum_mult",
    "p1_mean_mult",
    "p2_mean_mult",
    "p1_min_mult",
    "p2_min_mult",
    "p1_max_mult",
    "p2_max_mult",
    "p1_std_mult",
    "p2_std_mult",
    "diff_count_mult",
    "diff_sum_mult",
    "diff_mean_mult",
    "diff_min_mult",
    "diff_max_mult",
    "diff_std_mult",
]

# --- HELD ITEMS DICTIONARIES & METRIC PRIORS ---
JUNK_ITEMS = {
    "berryjuice", "berry", "noitem", "pokeball", "firestone",
    "energypowder", "aspearberry", "gengarite", "alakazite", "blackbelt"
}

STATUS_BERRIES = {
    "miracleberry", "mintberry", "przcureberry", "psncureberry",
    "bitterberry", "lumberry", "chestoberry"
}

TYPE_BOOST_ITEMS = {
    "charcoal", "magnet", "blackbelt", "pinkbow", "polkadotbow",
    "mysticwater", "hardstone", "poisonbarb", "softsand", "silverpowder",
    "blackglasses", "sharpbeak", "spelltag", "miracleseed", "twistedspoon",
    "nevermeltice", "dragonscale", "dragonfang", "metalcoat"
}

RNG_ITEMS = {
    "berserkgene", "scopelens", "quickclaw", "focusband", "brightpowder", "kingsrock"
}

HEALING_MOVES = {
    "rest", "recover", "softboiled", "milkdrink", "synthesis",
    "morningsun", "moonlight", "drainpunch", "megadrain", "gigadrain", "leechlife"
}


# --- FEATURE SELECTION ---
# Selection searches ALL_FEATURES; every listed column must be extracted.
# It takes priority over the manual ALLOWED_FEATURES filter.
USE_FEATURE_SELECTION = True
USE_ALLOWED_FEATURES = False
ALLOWED_FEATURES = []  # Paste a selected subset here for later manual runs.

# Shared settings for both runs.
SELECTION_MIN_FEATURES = 1
SELECTION_VALIDATION_FRACTION = 0.25

# Run 1: light exploration of all available ALL_FEATURES; use the settings below.
# Halving tests about 8 sizes for 135 features, 10 for 700, per search.
# Prefer fewer features within 0.2 percentage points of the best inner accuracy.
SELECTION_DROP_FRACTION = 0.5
SELECTION_MAX_ITERATIONS = 500  # Cheaper models for the search only.
SELECTION_SHAP_SAMPLE_SIZE = 512
SELECTION_ACCURACY_TOLERANCE = 0.002

# Run 2: deep, expensive refinement; first reduce ALL_FEATURES to ~20-40
# candidates from run 1 (winner plus promising alternatives). Extraction stays unchanged.
# USE_ALLOWED_FEATURES does not restrict this search; edit ALL_FEATURES itself.
# Suggested overrides; keep the other settings above:
# SELECTION_DROP_FRACTION = 0.1
# SELECTION_MAX_ITERATIONS = 2000
# SELECTION_SHAP_SAMPLE_SIZE = 2048
# SELECTION_ACCURACY_TOLERANCE = 0.001  # Within 0.1 percentage points of the best inner score.


# --- CATBOOST AND VALIDATION ---
RANDOM_STATE = 42
K_FOLDS = 5
N_REPEATS = 3
EARLY_STOPPING_ROUNDS = 100
EARLY_STOPPING_FRACTION = 0.15
DECISION_THRESHOLD = 0.5

# String/category columns are detected automatically. Add integer-coded
# categorical columns here when extending the feature lists.
CATEGORICAL_FEATURES = []

# Starting settings, not tuned results. Early stopping selects the tree count
# using Logloss; compare feature experiments using validation accuracy.
CATBOOST_PARAMS = {
    "iterations": 2000,
    "depth": 6,
    "learning_rate": 0.03,
    "l2_leaf_reg": 5.0,
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "task_type": "CPU",
    "thread_count": -1,
    "allow_writing_files": False,
    "verbose": False,
}

## 1. Loading Data

In [ ]:
# --- LOAD DATA ---
def load_data(file_path: str) -> list:
    """
    Loads the whole `file_path` and stores it as a list of battles
    """

    print(f"Loading data from '{file_path}'...")
    data = []
    try:
        with open(file_path, "r") as f:
            for line in f:
                data.append(json.loads(line))
        print(f"SUCCESS: loaded {len(data)} battles.")
        
    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")

    return data


# Loading data
train_data = load_data(TRAIN_FILE_PATH)
test_data  = load_data(TEST_FILE_PATH)

### Pokemon Stats & Moves Type

In [ ]:
# --- POKEMON TYPE MAP ---
def create_stats_map(train_data: list, test_data: list) -> dict:
    """
    Build a species stats/type lookup from supplied battle details, without labels
    """

    print("Creating Pokémon stats lookup...")
    stats_map = {}
    for data in (train_data, test_data):
        for battle in data:

            for pokemon in battle["p1_team_details"]:
                if pokemon["name"] not in stats_map:
                    stats_map[pokemon["name"]] = {"base_stats": {}, "types": []}
                stats_map[pokemon["name"]]["base_stats"] = {
                    "base_hp": pokemon["base_hp"],
                    "base_atk": pokemon["base_atk"],
                    "base_def": pokemon["base_def"],
                    "base_spa": pokemon["base_spa"],
                    "base_spd": pokemon["base_spd"],
                    "base_spe": pokemon["base_spe"],
                }
                stats_map[pokemon["name"]]["types"] = pokemon["types"]

            if battle["p2_lead_details"]["name"] not in stats_map:
                stats_map[battle["p2_lead_details"]["name"]] = {"base_stats": {}, "types": []}
            lead = battle["p2_lead_details"]
            stats_map[lead["name"]]["base_stats"] = {
                "base_hp": lead["base_hp"],
                "base_atk": lead["base_atk"],
                "base_def": lead["base_def"],
                "base_spa": lead["base_spa"],
                "base_spd": lead["base_spd"],
                "base_spe": lead["base_spe"],
            }
            stats_map[lead["name"]]["types"] = lead["types"]

    print(f"SUCCESS: found {len(stats_map)} pokemon.")
    return stats_map


# --- MOVES TYPE JSON ---
def normalize_type_chart(type_chart: pd.DataFrame) -> pd.DataFrame:
    """
    Return a copy with trimmed column names and lowercase, trimmed types; reject missing types.
    """

    # Clear columns headers
    type_chart.columns = type_chart.columns.astype(str).str.strip()

    # Clear columns values
    for col in ["attacking_type", "defending_type"]:
        # If a value is Nan break the run
        if type_chart[col].isna().any():
            nan_rows = type_chart[type_chart[col].isna()].index.tolist()
            raise ValueError(f"ERROR: found NaN value in column '{col}' at rows {nan_rows}.")

        # Remove white spaces and lower
        type_chart[col] = type_chart[col].astype(str).str.strip().str.lower()

    print(f"SUCCESS: cleaned {len(type_chart)} entries.")
    return type_chart

def load_type_chart(file_path: str) -> pd.DataFrame | None:
    """
    Load and normalize a type chart CSV
    """

    print(f"Loading data from '{file_path}'...")
    try:
        raw_df = pd.read_csv(file_path)
        return normalize_type_chart(raw_df)

    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")


# --- MULTPLIERS --- 
def get_multiplier(move_type: str, pkm_type1: str, pkm_type2: str) -> float:
    """
    Return the move's combined effectiveness against the pokemon's types
    """

    m1 = type_chart.get((move_type, pkm_type1), 1.0) if pkm_type1 else 1.0
    m2 = type_chart.get((move_type, pkm_type2), 1.0) if pkm_type2 else 1.0
    return m1 * m2


# Getting type charts
type_chart = load_type_chart(TYPE_CHART_PATH)
df_indexed = type_chart.set_index(["attacking_type", "defending_type"])
type_chart = df_indexed["multiplier"].to_dict()
stats_map = create_stats_map(train_data, test_data)

## 2. Extract Features (both Train and Test Features)

In [ ]:
# --- BASE STATS ---
def get_stats(turn: dict, p2_stats: dict) -> None:
    """
    Store player 2's active pokemon stats in p2_stats if found in stats_map
    """

    pkm2_name = turn["p2_pokemon_state"]["name"]
    if pkm2_name in stats_map:
        stats = stats_map[pkm2_name]["base_stats"]
        p2_stats[pkm2_name] = stats

def feature_base_stats(battle: dict, p2_stats: dict, features: dict) -> None:
    """
    Add base pokemon statistics:
    - player 1 full team absolute stats (measures team building quality)
    - player 2 seen team absolute stats (measures active counters deployed)
    - differential stats (P1 - P2: higher = P1 advantage)
    """

    for player in ("p1", "p2"):
        if player == "p1":
            team = battle["p1_team_details"]
        else:
            team = p2_stats.values()

        # Single values + merged values
        features[f"{player}_sum_bst"] = 0
        features[f"{player}_mean_bst"] = 0
        bst_values = np.zeros(len(team))  # One total per Pokemon with known stats
        for stat in STATISTICS:
            values = [pokemon[f"base_{stat}"] for pokemon in team]
            features[f"{player}_sum_{stat}"]  = np.sum(values)
            features[f"{player}_mean_{stat}"] = np.mean(values)
            features[f"{player}_min_base_{stat}"] = min(values)
            features[f"{player}_max_base_{stat}"] = max(values)
            features[f"{player}_std_base_{stat}"] = float(np.std(values, ddof=0))

            features[f"{player}_sum_bst"] += features[f"{player}_sum_{stat}"]
            bst_values += values

        # Merged values
        features[f"{player}_mean_bst"] = features[f"{player}_sum_bst"] / TEAM_SIZE
        features[f"{player}_std_base_bst"] = float(np.std(bst_values, ddof=0))

    # Differentials
    for stat in STATISTICS:
        features[f"diff_sum_{stat}"]  = features[f"p1_sum_{stat}"] - features[f"p2_sum_{stat}"]
        features[f"diff_mean_{stat}"] = features[f"p1_mean_{stat}"] - features[f"p2_mean_{stat}"]
    features["diff_sum_bst"]  = features["p1_sum_bst"]  - features["p2_sum_bst"]
    features["diff_mean_bst"] = features["p1_mean_bst"] - features["p2_mean_bst"]
    features["diff_std_base_bst"] = features["p1_std_base_bst"] - features["p2_std_base_bst"]


# --- HP LOSS ---
def get_hp_pct_lost(turn: dict, p1_hp_map: dict, p2_hp_map: dict) -> None:
    """
    Track HP loss from one turn for each observed pokemon:
    - remember its latest HP across switches, starting from full HP
    - accumulate HP loss in its three ten-turn intervals
    Healing reduces HP loss.
    """

    name = turn["p1_pokemon_state"]["name"]
    hp = turn["p1_pokemon_state"]["hp_pct"]
    if name not in p1_hp_map:
        p1_hp_map[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}
    pokemon = p1_hp_map[name]
    pokemon["losses"][(turn["turn"] - 1) // 10] += pokemon["last_hp"] - hp
    pokemon["last_hp"] = hp

    name = turn["p2_pokemon_state"]["name"]
    hp = turn["p2_pokemon_state"]["hp_pct"]
    if name not in p2_hp_map:
        p2_hp_map[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}
    pokemon = p2_hp_map[name]
    pokemon["losses"][(turn["turn"] - 1) // 10] += pokemon["last_hp"] - hp
    pokemon["last_hp"] = hp

def feature_hp_lost(p1_hp_map: dict, p2_hp_map: dict, features: dict) -> None:
    """
    Add net HP loss in each ten-turn interval and in total, divided by TEAM_SIZE.
    Unseen pokemon contribute zero loss; healing reduces loss.
    Differentials use P2 loss - P1 loss: higher = P1 advantage.
    """

    for player in ("p1", "p2"):
        if player == "p1":
            team = p1_hp_map
        else:
            team = p2_hp_map

        # Single values + merged values
        features[f"{player}_sum_hp_lost"] = 0
        # Keep one total per Pokemon, with zero loss for unseen Pokemon.
        hp_lost_values = np.zeros(TEAM_SIZE)
        for i, interval in enumerate(HP_PCT_LOST_INTERVALS):
            values = [pokemon["losses"][i] for pokemon in team.values()]
            features[f"{player}_hp_pct_lost_{interval}"] = sum(values)
            features[f"{player}_sum_hp_lost"] += features[f"{player}_hp_pct_lost_{interval}"]
            hp_lost_values[:len(team)] += values

        # Merged values
        features[f"{player}_mean_hp_pct_lost"] = features[f"{player}_sum_hp_lost"] / TEAM_SIZE
        features[f"{player}_std_hp_pct_lost"] = float(np.std(hp_lost_values, ddof=0))

    # Differentials
    for interval in HP_PCT_LOST_INTERVALS:
        features[f"diff_hp_pct_lost_{interval}"] = features[f"p2_hp_pct_lost_{interval}"] - features[f"p1_hp_pct_lost_{interval}"]
    features["diff_sum_hp_pct_lost"]  = features["p2_sum_hp_lost"] - features["p1_sum_hp_lost"]
    features["diff_mean_hp_pct_lost"] = features["p2_mean_hp_pct_lost"] - features["p1_mean_hp_pct_lost"]
    features["diff_std_hp_pct_lost"]  = features["p2_std_hp_pct_lost"] - features["p1_std_hp_pct_lost"]


# --- TYPE EFFECTIVENESS ---
def get_mults(turn: dict, p1_mults: dict, p2_mults: dict) -> None:
    """
    Update both players' type effectiveness multiplier counts for one turn (damaging moves only).
    """

    mv1 = turn["p1_move_details"]
    if mv1 is not None and mv1["category"] != "status":
        mv1_type = mv1["type"]
        pkm2_name = turn["p2_pokemon_state"]["name"]
        pkm2_stats = stats_map.get(pkm2_name)
        if pkm2_stats is not None:
            pkm2_types = pkm2_stats.get("types", ["notype", "notype"])
        else:
            pkm2_types = ["notype", "notype"]
        mult1 = get_multiplier(mv1_type, pkm2_types[0], pkm2_types[1])
        p1_mults[mult1] += 1

    mv2 = turn["p2_move_details"]
    if mv2 is not None and mv2["category"] != "status":
        mv2_type = mv2["type"]
        pkm1_name = turn["p1_pokemon_state"]["name"]
        pkm1_types = stats_map[pkm1_name]["types"]
        mult2 = get_multiplier(mv2_type, pkm1_types[0], pkm1_types[1])
        p2_mults[mult2] += 1

def feature_mults(p1_mults: dict, p2_mults: dict, features: dict) -> None:
    """
    Add bucket counts and population summaries of recorded attack multipliers.
    Count advantage is P1 - P2 above 1x and P2 - P1 below 1x; 1x is neutral.
    """

    for player in ("p1", "p2"):
        if player == "p1":
            team = p1_mults
        else:
            team = p2_mults

        # Single values + merged values
        values = []
        features[f"{player}_count_mult"] = 0
        features[f"{player}_sum_mult"] = 0
        for mult in MULTIPLIERS:
            value = team[mult]
            values.extend([mult] * value)
            features[f"{player}_{mult}_mult"] = value
            features[f"{player}_sum_mult"] += mult * value
            features[f"{player}_count_mult"] += value

        # Merged values
        features[f"{player}_min_mult"] = min(values) if values else np.nan
        features[f"{player}_max_mult"] = max(values) if values else np.nan
        features[f"{player}_mean_mult"] = features[f"{player}_sum_mult"] / features[f"{player}_count_mult"] if values else np.nan
        features[f"{player}_std_mult"] = float(np.std(values, ddof=0)) if values else np.nan

    # Differentials
    features["diff_count_mult"] = 0
    for mult in MULTIPLIERS:
        if mult < 1:
            features[f"diff_{mult}_mult"] = features[f"p2_{mult}_mult"] - features[f"p1_{mult}_mult"]
            features["diff_count_mult"] += features[f"diff_{mult}_mult"]
        else:
            features[f"diff_{mult}_mult"] = features[f"p1_{mult}_mult"] - features[f"p2_{mult}_mult"]
            features["diff_count_mult"] += features[f"diff_{mult}_mult"]
    features["diff_min_mult"] = features["p1_min_mult"] - features["p2_min_mult"]
    features["diff_max_mult"] = features["p1_max_mult"] - features["p2_max_mult"]
    features["diff_sum_mult"]  = features["p1_sum_mult"] - features["p2_sum_mult"]
    features["diff_mean_mult"] = features["p1_mean_mult"] - features["p2_mean_mult"]
    features["diff_std_mult"] = features["p1_std_mult"] - features["p2_std_mult"]


# --- STATUSES (ALSO FAINT) ---
def get_statuses(turn: dict, p1_status: dict, p2_status: dict) -> None:
    """
    Accumulate active status turn counts for both players, including fainting
    """
    
    pkm1_state = turn["p1_pokemon_state"]["status"]
    if pkm1_state != "nostatus":
        if pkm1_state not in p1_status:
            p1_status[pkm1_state] = 0
        p1_status[pkm1_state] += 1

    pkm2_state = turn["p2_pokemon_state"]["status"]
    if pkm2_state != "nostatus":
        if pkm2_state not in p2_status:
            p2_status[pkm2_state] = 0
        p2_status[pkm2_state] += 1

def feature_statuses(p1_status_map: dict, p2_status_map: dict, n_turns: int, features: dict) -> None:
    """
    Add status turn totals, weighted sums, and differentials.
    Pre-initializes all known statuses to eliminate train/test schema mismatches.
    """

    features["p1_statuses_wsum"] = 0
    features["p2_statuses_wsum"] = 0

    # Pre-initialize all statuses from STATUSES_WEIGHTS
    for status, weight in STATUSES_WEIGHTS.items():
        val_sts_1 = p1_status_map.get(status, 0)
        val_sts_2 = p2_status_map.get(status, 0)
        features[f"p1_{status}"] = val_sts_1
        features[f"p2_{status}"] = val_sts_2
        features[f"diff_{status}"] = val_sts_2 - val_sts_1
        features["p1_statuses_wsum"] += val_sts_1 * weight
        features["p2_statuses_wsum"] += val_sts_2 * weight

    features["p1_statuses_wmean"] = features["p1_statuses_wsum"] / n_turns
    features["p2_statuses_wmean"] = features["p2_statuses_wsum"] / n_turns

    # Differentials: P2 minus P1 (higher = P2 suffered more status afflictions = P1 advantage)
    features["diff_statuses_wsum"]  = features["p2_statuses_wsum"] - features["p1_statuses_wsum"]
    features["diff_statuses_wmean"] = features["p2_statuses_wmean"] - features["p1_statuses_wmean"]


# --- BOOSTS (CONSISTENT SCHEMA & DIFFERENTIALS) ---
def get_boosts(turn: dict, p1_boosts: dict, p2_boosts: dict) -> None:
    """
    Accumulate signed boost stages per stat for both active pokemon.
    Update both players' totals in place using the current observed turn
    """

    pkm1_boosts = turn["p1_pokemon_state"]["boosts"]
    for boost, value in pkm1_boosts.items():
        p1_boosts[boost] += value

    pkm2_boosts = turn["p2_pokemon_state"]["boosts"]
    for boost, value in pkm2_boosts.items():
        p2_boosts[boost] += value

def feature_boosts(p1_boosts: dict, p2_boosts: dict, n_turns: int, features: dict) -> None:
    """
    Add signed boost stage totals per stat for both players, weighted sums,
    and differentials favoring P1.
    Pre-initializes all stats to eliminate train/test schema mismatches.
    """

    features["p1_boosts_wsum"] = 0
    features["p2_boosts_wsum"] = 0

    # Pre-initialize all boosts from BOOSTS_WEIGHTS
    for boost, weight in BOOSTS_WEIGHTS.items():
        val_bst_1 = p1_boosts.get(boost, 0)
        val_bst_2 = p2_boosts.get(boost, 0)
        features[f"p1_{boost}_boost"] = val_bst_1
        features[f"p2_{boost}_boost"] = val_bst_2
        features[f"diff_{boost}_boost"] = val_bst_1 - val_bst_2
        features["p1_boosts_wsum"] += val_bst_1 * weight
        features["p2_boosts_wsum"] += val_bst_2 * weight

    features["p1_boosts_wmean"] = features["p1_boosts_wsum"] / n_turns
    features["p2_boosts_wmean"] = features["p2_boosts_wsum"] / n_turns

    # Differentials (P1 - P2: higher = P1 has more positive stat stages)
    features["diff_boosts_wsum"]  = features["p1_boosts_wsum"] - features["p2_boosts_wsum"]
    features["diff_boosts_wmean"] = features["p1_boosts_wmean"] - features["p2_boosts_wmean"]


# --- EFFECTS FEATURES ---
def get_effects(turn: dict, p1_effects: dict, p2_effects: dict) -> None:
    """
    Accumulate positive and negative effect weights from one turn for both players
    """

    for eff in turn["p1_pokemon_state"]["effects"]:
        w = EFFECT_WEIGHTS.get(eff, 0.0)
        if w > 0:
            p1_effects["pos"] += w
        elif w < 0:
            p1_effects["neg"] += w

    for eff in turn["p2_pokemon_state"]["effects"]:
        w = EFFECT_WEIGHTS.get(eff, 0.0)
        if w > 0:
            p2_effects["pos"] += w
        elif w < 0:
            p2_effects["neg"] += w

def feature_effects(p1_effects: dict, p2_effects: dict, n_turns: int, features: dict) -> None:
    """
    Add weighted effect scores after all observed turns and at the final turn:
    - mean positive, negative, and net effect weights for P1 and P2
    - net weighted effect difference (P1 - P2) across the battle
    """

    features["p1_eff_pos_mean"] = p1_effects["pos"] / n_turns
    features["p1_eff_neg_mean"] = p1_effects["neg"] / n_turns
    features["p2_eff_pos_mean"] = p2_effects["pos"] / n_turns
    features["p2_eff_neg_mean"] = p2_effects["neg"] / n_turns

    features["p1_eff_net_mean"]   = features["p1_eff_pos_mean"] + features["p1_eff_neg_mean"]
    features["p2_eff_net_mean"]   = features["p2_eff_pos_mean"] + features["p2_eff_neg_mean"]

    features["diff_pos_mean"]     = features["p1_eff_pos_mean"] - features["p2_eff_pos_mean"]
    features["diff_neg_mean"]     = features["p2_eff_neg_mean"] - features["p1_eff_neg_mean"]
    features["diff_eff_net_mean"] = features["p1_eff_net_mean"] - features["p2_eff_net_mean"]


# --- TEAM SNAPSHOT AT LAST TURN ---
def get_last_seen(turn: dict, p1_last_seen: dict, p2_last_seen: dict) -> None:
    """
    Store each observed pokemon's latest HP and status stages.
    Update both players' maps in place; retain switched-out pokemon's snapshots.
    """

    pkm1 = turn["p1_pokemon_state"]
    p1_last_seen[pkm1["name"]] = {
        "hp_pct": pkm1["hp_pct"],
        "status": pkm1["status"]
    }

    pkm2 = turn["p2_pokemon_state"]
    p2_last_seen[pkm2["name"]] = {
        "hp_pct": pkm2["hp_pct"],
        "status": pkm2["status"]
    }

def feature_last_seen(last_turn: dict, p1_last_seen: dict, p2_last_seen: dict, features: dict) -> None:
    """
    Add latest-known team HP, status, boost, and effect means and P1 - P2 differences.
    Divide by six; assume unseen pokemon have full HP and no status, effects, or boosts.
    Benched pokemon retain their last observed data, which may be older.
    """

    features["p1_lst_hp_mean"] = 1.0 * (TEAM_SIZE - len(p1_last_seen))
    features["p1_lst_statuses_wmean"] = 0.0
    features["p1_lst_effects_wmean"] = 0.0
    features["p1_lst_boosts_wmean"] = 0.0
    for pkm1 in p1_last_seen.values():
        features["p1_lst_hp_mean"] += pkm1["hp_pct"]
        features["p1_lst_statuses_wmean"] += STATUSES_WEIGHTS.get(pkm1["status"], 0)
    lst_pkm1_effects = last_turn["p1_pokemon_state"]["effects"]
    for eff in lst_pkm1_effects:
        features["p1_lst_effects_wmean"] += EFFECT_WEIGHTS.get(eff, 0)
    lst_pkm1_boosts = last_turn["p1_pokemon_state"]["boosts"]
    for bst, value in lst_pkm1_boosts.items():
        features["p1_lst_boosts_wmean"] += value * BOOSTS_WEIGHTS.get(bst, 0)
    features["p1_lst_hp_mean"]        /= TEAM_SIZE
    features["p1_lst_statuses_wmean"] /= TEAM_SIZE
    features["p1_lst_effects_wmean"]  /= TEAM_SIZE
    features["p1_lst_boosts_wmean"]   /= TEAM_SIZE

    features["p2_lst_hp_mean"] = 1.0 * (TEAM_SIZE - len(p2_last_seen))
    features["p2_lst_statuses_wmean"] = 0.0
    features["p2_lst_effects_wmean"] = 0.0
    features["p2_lst_boosts_wmean"] = 0.0
    for pkm2 in p2_last_seen.values():
        features["p2_lst_hp_mean"] += pkm2["hp_pct"]
        features["p2_lst_statuses_wmean"] += STATUSES_WEIGHTS.get(pkm2["status"], 0)
    lst_pkm2_effects = last_turn["p2_pokemon_state"]["effects"]
    for eff in lst_pkm2_effects:
        features["p2_lst_effects_wmean"] += EFFECT_WEIGHTS.get(eff, 0)
    lst_pkm2_boosts = last_turn["p2_pokemon_state"]["boosts"]
    for bst, value in lst_pkm2_boosts.items():
        features["p2_lst_boosts_wmean"] += value * BOOSTS_WEIGHTS.get(bst, 0)
    features["p2_lst_hp_mean"]        /= TEAM_SIZE
    features["p2_lst_statuses_wmean"] /= TEAM_SIZE
    features["p2_lst_effects_wmean"]  /= TEAM_SIZE
    features["p2_lst_boosts_wmean"]   /= TEAM_SIZE

    features["diff_lst_hp_mean"]        = features["p1_lst_hp_mean"] - features["p2_lst_hp_mean"]
    features["diff_lst_statuses_wmean"] = features["p2_lst_statuses_wmean"] - features["p1_lst_statuses_wmean"]
    features["diff_lst_effects_wmean"]  = features["p1_lst_effects_wmean"] - features["p2_lst_effects_wmean"]
    features["diff_lst_boosts_wmean"]   = features["p1_lst_boosts_wmean"] - features["p2_lst_boosts_wmean"]


# --- HELD ITEMS ---
def get_items(turn: dict, p2_item_tracker: dict) -> None:
    """
    Track observed P2 item activations turn by turn:
    - detects Leftovers recovery (approx 1/16 = 6.25% HP tick without healing moves)
    - detects item signatures appearing in turn effects
    """

    p2_state = turn.get("p2_pokemon_state", {})
    p2_mon = p2_state.get("name")
    curr_hp = p2_state.get("hp_pct", 1.0)
    p2_move = turn.get("p2_move_details")
    move_name = p2_move.get("name") if p2_move else ""

    # Check Leftovers passive regeneration between consecutive turns of the same active Pokemon
    if p2_item_tracker["prev_mon"] == p2_mon and p2_item_tracker["prev_hp"] is not None:
        hp_diff = curr_hp - p2_item_tracker["prev_hp"]
        if 0.04 <= hp_diff <= 0.08 and move_name not in HEALING_MOVES:
            p2_item_tracker["leftovers_seen"].add(p2_mon)

    # Check item names explicitly exposed in turn effects
    for eff in p2_state.get("effects", []):
        if eff in JUNK_ITEMS:
            p2_item_tracker["junk_seen"].add((p2_mon, eff))
        elif eff in RNG_ITEMS:
            p2_item_tracker["rng_seen"].add((p2_mon, eff))

    p2_item_tracker["prev_mon"] = p2_mon
    p2_item_tracker["prev_hp"] = curr_hp

def feature_items(battle: dict, p2_item_tracker: dict, features: dict) -> None:
    """
    Extract Player 1 item stats, Player 2 timeline detections, and their differentials
    """

    p1_team = battle.get("p1_team_details", [])
    p1_items = [p.get("item") or "noitem" for p in p1_team]

    # 1. Macro team item metrics
    p1_leftovers = p1_items.count("leftovers")
    features["p1_item_leftovers_count"]   = p1_leftovers
    features["p1_item_has_all_leftovers"] = int(p1_leftovers == 6)
    features["p1_item_unique_count"]      = len(set(p1_items))
    features["p1_item_non_leftovers"]     = 6 - p1_leftovers

    # 2. Archetype groupings
    p1_junk = sum(1 for it in p1_items if it in JUNK_ITEMS)
    features["p1_item_junk_count"]         = p1_junk
    features["p1_item_status_berry_count"] = sum(1 for it in p1_items if it in STATUS_BERRIES)
    features["p1_item_type_boost_count"]   = sum(1 for it in p1_items if it in TYPE_BOOST_ITEMS)
    features["p1_item_rng_count"]          = sum(1 for it in p1_items if it in RNG_ITEMS)

    # 3. High-impact item synergies
    features["p1_has_thickclub_marowak"] = int(any(
        p.get("name") in {"marowak", "cubone"} and p.get("item") == "thickclub"
        for p in p1_team
    ))
    features["p1_has_lightball_pikachu"] = int(any(
        p.get("name") == "pikachu" and p.get("item") == "lightball"
        for p in p1_team
    ))

    # 4. Player 2 timeline detections and differentials
    p2_leftovers = len(p2_item_tracker["leftovers_seen"])
    p2_junk = len(p2_item_tracker["junk_seen"])

    features["p2_detected_leftovers_count"] = p2_leftovers
    features["diff_leftovers_count"]        = p1_leftovers - p2_leftovers
    features["diff_junk_items_count"]       = p1_junk - p2_junk


# --- DEPLOYED LEVELS & EFFECTIVE HP (OUTLIER-AWARE & BASE HP x LEVEL x HP% CORRELATION) ---
def calc_eff_stat(base_val: float, level: int, is_hp: bool = False) -> float:
    """
    Compute the Gen 2 effective stat from base stat and level (assuming standard max DV=15 and max StatExp).
    Correlates base stats with Pokemon level directly.
    """

    core = ((base_val + 47) * 2 * level) / 100.0
    return core + level + 10 if is_hp else core + 5

def get_deployed_turns(turn: dict, p1_deployed_turns: dict, p2_deployed_turns: dict) -> None:
    """
    Count the number of active turns spent on the field by each deployed Pokemon.
    """

    p1_name = turn["p1_pokemon_state"]["name"]
    p1_deployed_turns[p1_name] = p1_deployed_turns.get(p1_name, 0) + 1

    p2_name = turn["p2_pokemon_state"]["name"]
    p2_deployed_turns[p2_name] = p2_deployed_turns.get(p2_name, 0) + 1

def feature_deployed_and_levels(
    battle: dict,
    p1_deployed_turns: dict,
    p2_deployed_turns: dict,
    p1_hp_map: dict,
    p2_hp_map: dict,
    features: dict
) -> None:
    """
    Add high-signal, low-collinearity features tailored for the 99.4% Lvl 100 / 0.6% Underleveled distribution:
    1. P1 & P2 deployed counts, differential deployed count, and turn-weighted active level
    2. Remaining Effective HP Ratio at Turn 30 (correlates base_hp, actual level, and latest hp_pct)
    """

    p1_team = battle["p1_team_details"]
    p1_team_map = {p["name"]: p for p in p1_team}
    n_turns = max(sum(p1_deployed_turns.values()), 1)

    deployed_pkm = [p1_team_map[name] for name in p1_deployed_turns if name in p1_team_map]

    # 1. Specific indicators for level outliers (< 100) and field presence symmetry
    features["p1_deployed_count"]              = len(deployed_pkm)
    features["p2_deployed_count"]              = len(p2_deployed_turns)
    features["diff_deployed_count"]            = features["p2_deployed_count"] - features["p1_deployed_count"]
    features["p1_underleveled_count"]          = sum(1 for p in p1_team if p["level"] < 100)
    features["p1_deployed_underleveled_count"] = sum(1 for p in deployed_pkm if p["level"] < 100)
    features["p1_team_min_lvl"]                = np.min([p["level"] for p in p1_team])
    features["p1_deployed_min_lvl"]            = np.min([p["level"] for p in deployed_pkm]) if deployed_pkm else 100.0
    features["p1_active_wmean_lvl"]            = sum(
        p1_team_map[name]["level"] * turns
        for name, turns in p1_deployed_turns.items()
        if name in p1_team_map
    ) / n_turns

    # 2. Remaining Effective HP Ratio at Turn 30 (correlates Base HP, real Level, and remaining HP%)
    p1_rem_eff_hp, p1_tot_eff_hp = 0.0, 0.0
    for p in p1_team:
        eff_hp = calc_eff_stat(p["base_hp"], p["level"], is_hp=True)
        last_hp_pct = p1_hp_map[p["name"]]["last_hp"] if p["name"] in p1_hp_map else 1.0
        p1_rem_eff_hp += eff_hp * last_hp_pct
        p1_tot_eff_hp += eff_hp
    features["p1_rem_effective_hp_ratio"] = p1_rem_eff_hp / p1_tot_eff_hp if p1_tot_eff_hp > 0 else 0.0

    # For P2: use exact level for the lead, and 100 (99.4% mode) for other seen team members
    p2_lead = battle["p2_lead_details"]
    p2_rem_eff_hp, p2_tot_eff_hp = 0.0, 0.0
    for name in p2_deployed_turns:
        if name == p2_lead["name"]:
            base_hp, lvl = p2_lead["base_hp"], p2_lead["level"]
        elif name in stats_map:
            base_hp, lvl = stats_map[name]["base_stats"]["base_hp"], 100
        else:
            base_hp, lvl = 80, 100
        eff_hp = calc_eff_stat(base_hp, lvl, is_hp=True)
        last_hp_pct = p2_hp_map[name]["last_hp"] if name in p2_hp_map else 1.0
        p2_rem_eff_hp += eff_hp * last_hp_pct
        p2_tot_eff_hp += eff_hp

    # Unseen P2 Pokemon are assumed at 100% HP with standard OU bulk (base_hp=85, lvl=100)
    unseen_p2_count = max(TEAM_SIZE - len(p2_deployed_turns), 0)
    default_p2_eff_hp = calc_eff_stat(85, 100, is_hp=True)
    p2_rem_eff_hp += unseen_p2_count * default_p2_eff_hp
    p2_tot_eff_hp += unseen_p2_count * default_p2_eff_hp
    features["p2_rem_effective_hp_ratio"] = p2_rem_eff_hp / p2_tot_eff_hp if p2_tot_eff_hp > 0 else 0.0

    features["diff_rem_effective_hp_ratio"] = (
        features["p1_rem_effective_hp_ratio"] - features["p2_rem_effective_hp_ratio"]
    )


# --- TACTICAL TEMPO, OFFENSIVE POWER, HAZARDS & REST ---
def get_tactics(turn: dict, p1_tactics: dict, p2_tactics: dict) -> None:
    """
    Track turn-by-turn tactical momentum for both players:
    - voluntary switches (excluding post-KO replacements)
    - distinct fainted Pokemon
    - effective offensive power (Base Power * STAB * Type Multiplier * Accuracy)
    - Rest / Sleep Talk usage and Spikes / Phazing (Roar/Whirlwind)
    """
    for side, opp_side, tracker, opp_tracker in (
        ("p1", "p2", p1_tactics, p2_tactics),
        ("p2", "p1", p2_tactics, p1_tactics),
    ):
        state = turn[f"{side}_pokemon_state"]
        opp_state = turn[f"{opp_side}_pokemon_state"]
        mv = turn[f"{side}_move_details"]
        name = state["name"]
        is_fainted = (state["status"] == "fnt" or state["hp_pct"] == 0.0)

        # Conta solo i cambi volontari (esclude gli ingressi forzati dopo un KO al turno precedente)
        if tracker["prev_mon"] is not None and tracker["prev_mon"] != name:
            if not tracker.get("prev_fainted", False):
                tracker["switches"] += 1
        tracker["prev_mon"] = name
        tracker["prev_fainted"] = is_fainted

        # Track distinct fainted Pokemon
        if is_fainted:
            tracker["fainted_mons"].add(name)

        if mv is not None:
            tracker["moves_used"] += 1
            mv_name = mv["name"]
            mv_type = mv["type"]

            # Azioni di stato valide solo se il Pokemon non è andato KO prima di eseguirle
            if not is_fainted:
                if mv_name == "rest":
                    tracker["rest_used"] += 1
                elif mv_name == "sleeptalk":
                    tracker["sleeptalk_used"] += 1
                elif mv_name == "spikes":
                    tracker["spikes_active"] = 1
                elif mv_name == "rapidspin":
                    opp_tracker["spikes_active"] = 0
                elif mv_name in {"roar", "whirlwind"}:
                    tracker["phaze_used"] += 1

            # Effective offensive power for damaging moves
            if mv["category"] != "status" and mv["base_power"] > 0:
                attacker_types = stats_map.get(name, {}).get("types", ["notype", "notype"])
                defender_types = stats_map.get(opp_state["name"], {}).get("types", ["notype", "notype"])
                stab = 1.5 if mv_type in attacker_types else 1.0
                mult = get_multiplier(mv_type, defender_types[0], defender_types[1])
                acc = mv.get("accuracy") or 1.0
                tracker["power_sum"] += mv["base_power"] * stab * mult * acc


def feature_tactics(p1_tactics: dict, p2_tactics: dict, n_turns: int, features: dict) -> None:
    """Compute tactical differentials from accumulated timeline trackers."""
    # 1. True distinct fainted counts & alive differential
    p1_ko = len(p1_tactics["fainted_mons"])
    p2_ko = len(p2_tactics["fainted_mons"])
    features["p1_distinct_ko"]     = p1_ko
    features["p2_distinct_ko"]     = p2_ko
    features["diff_pokemon_alive"] = p2_ko - p1_ko

    # 2. Action tempo & switching
    features["p1_moves_used"]   = p1_tactics["moves_used"] / n_turns
    features["p2_moves_used"]   = p2_tactics["moves_used"] / n_turns
    features["diff_moves_used"] = (p1_tactics["moves_used"] - p2_tactics["moves_used"]) / n_turns
    features["p1_switches"]     = p1_tactics["switches"] / n_turns
    features["p2_switches"]     = p2_tactics["switches"] / n_turns
    features["diff_switches"]   = (p1_tactics["switches"] - p2_tactics["switches"]) / n_turns

    # 3. Effective offensive power per turn
    features["p1_eff_power_mean"]   = p1_tactics["power_sum"] / n_turns
    features["p2_eff_power_mean"]   = p2_tactics["power_sum"] / n_turns
    features["diff_eff_power_mean"] = features["p1_eff_power_mean"] - features["p2_eff_power_mean"]

    # 4. Gen 2 Meta: RestTalk & Spikes/Phazing
    features["p1_rest_used"]        = p1_tactics["rest_used"]
    features["p2_rest_used"]        = p2_tactics["rest_used"]
    features["diff_rest_used"]      = p1_tactics["rest_used"] - p2_tactics["rest_used"]
    features["p1_sleeptalk_used"]   = p1_tactics["sleeptalk_used"]
    features["p2_sleeptalk_used"]   = p2_tactics["sleeptalk_used"]
    features["diff_sleeptalk_used"] = p1_tactics["sleeptalk_used"] - p2_tactics["sleeptalk_used"]
    features["p1_spikes_active"]    = p1_tactics["spikes_active"]
    features["p2_spikes_active"]    = p2_tactics["spikes_active"]
    features["diff_spikes_active"]  = p1_tactics["spikes_active"] - p2_tactics["spikes_active"]
    features["p1_phaze_used"]       = p1_tactics["phaze_used"]
    features["p2_phaze_used"]       = p2_tactics["phaze_used"]
    features["diff_phaze_used"]     = p1_tactics["phaze_used"] - p2_tactics["phaze_used"]

In [ ]:
# --- EXTRACT FEATURES ---
def extract_features(data: list) -> pd.DataFrame:
    """
    Extract comprehensive features combining absolute scale and differentials:
    1. base stats (P1 absolute, P2 absolute, diff P1 - P2)
    2. HP loss in intervals and total (P1, P2, and differentials)
    3. type effectiveness multiplier counts and differentials
    4. statuses (pre-initialized, consistent schema, and differentials)
    5. boosts (P1, P2, and differentials)
    6. battle effects (P1, P2, and differentials)
    7. last-turn snapshot at Turn 30 (P1, P2, and differentials)
    8. held items (P1 known items, P2 timeline detections, and differentials)
    9. deployed Pokemon levels, symmetry counts & effective HP ratios
    10. tactical tempo, effective offensive power, hazards, phazing & RestTalk
    """

    print("Extracting features...")
    feature_list = []

    for battle in data:
        features = {"battle_id" : battle["battle_id"]}

        # Keep the identifier and target for the training and submission cells
        if "player_won" in battle:
            features["player_won"] = battle["player_won"]


        # Initializations
        p1_hp_map, p2_hp_map = {}, {}
        p1_mults = {mult: 0 for mult in MULTS_WEIGHTS}
        p2_mults = {mult: 0 for mult in MULTS_WEIGHTS}
        p1_status, p2_status = {}, {}
        p1_boosts = {boost: 0 for boost in BOOSTS_WEIGHTS}
        p2_boosts = {boost: 0 for boost in BOOSTS_WEIGHTS}
        p1_effects = {"pos": 0.0, "neg": 0.0}
        p2_effects = {"pos": 0.0, "neg": 0.0}
        p1_last_seen, p2_last_seen = {}, {}
        p2_stats = {}
        p2_item_tracker = {
            "leftovers_seen": set(),
            "junk_seen": set(),
            "rng_seen": set(),
            "prev_mon": None,
            "prev_hp": None,
        }
        p1_deployed_turns, p2_deployed_turns = {}, {}
        p1_tactics = {
            "prev_mon": None, "prev_fainted": False, "switches": 0, "fainted_mons": set(), "moves_used": 0,
            "rest_used": 0, "sleeptalk_used": 0, "spikes_active": 0, "phaze_used": 0, "power_sum": 0.0
        }
        p2_tactics = {
            "prev_mon": None, "prev_fainted": False, "switches": 0, "fainted_mons": set(), "moves_used": 0,
            "rest_used": 0, "sleeptalk_used": 0, "spikes_active": 0, "phaze_used": 0, "power_sum": 0.0
        }

        timeline = battle["battle_timeline"]
        for turn in timeline:
            get_stats(turn, p2_stats)
            get_hp_pct_lost(turn, p1_hp_map, p2_hp_map)
            get_mults(turn, p1_mults, p2_mults)
            get_statuses(turn, p1_status, p2_status)
            get_boosts(turn, p1_boosts, p2_boosts)
            get_effects(turn, p1_effects, p2_effects)
            get_last_seen(turn, p1_last_seen, p2_last_seen)
            get_items(turn, p2_item_tracker)
            get_deployed_turns(turn, p1_deployed_turns, p2_deployed_turns)
            get_tactics(turn, p1_tactics, p2_tactics)

        last_turn = timeline[-1]
        # 1. Base stats
        feature_base_stats(battle, p2_stats, features)
        # 2. HP loss in intervals, totals & differentials
        feature_hp_lost(p1_hp_map, p2_hp_map, features)
        # 3. Type effectiveness multiplier counts & differentials
        feature_mults(p1_mults, p2_mults, features)
        # 4. Statuses & differentials
        # feature_statuses(p1_status, p2_status, len(timeline), features)
        # 5. Boosts & differentials
        # feature_boosts(p1_boosts, p2_boosts, len(timeline), features)
        # 6. Effects & differentials
        # feature_effects(p1_effects, p2_effects, len(timeline), features)
        # 7. Last seen & differentials
        # feature_last_seen(last_turn, p1_last_seen, p2_last_seen, features)
        # 8. Items
        # feature_items(battle, p2_item_tracker, features)
        # 9. Deployed Pokemon levels & effective stats (Base Stats x Level)
        # feature_deployed_and_levels(battle, p1_deployed_turns, p2_deployed_turns, p1_hp_map, p2_hp_map, features)
        # 10. Tactical tempo, offensive power, hazards & RestTalk
        # feature_tactics(p1_tactics, p2_tactics, len(timeline), features)

        feature_list.append(features)

    print(f"SUCCESS: extracted {len(feature_list[0])} features.")
    return pd.DataFrame(feature_list).fillna(0)


# Extracting training and test features
train_df = extract_features(train_data)
print(f"Training features: {train_df.shape}.")
display(train_df.head())

test_df = extract_features(test_data)
print(f"Test features: {test_df.shape}.")
# display(test_df.head())

## 3. Training

### Models

In [ ]:
def build_model( random_state: int = RANDOM_STATE, iterations: int | None = None ) -> CatBoostClassifier:
    """
    Create a fresh CatBoost classifier with the configured settings
    """

    params = {**CATBOOST_PARAMS, "random_seed": random_state}
    if iterations is not None:
        params["iterations"] = iterations
    return CatBoostClassifier(**params)


def prepare_features(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple:
    """
    Align features and normalize categorical values without using labels
    """

    if USE_FEATURE_SELECTION:
        feature_cols = list(dict.fromkeys(ALL_FEATURES))
    elif USE_ALLOWED_FEATURES:
        feature_cols = list(dict.fromkeys(ALLOWED_FEATURES))
    else:
        feature_cols = [
            col for col in train_df.columns
            if col not in {"battle_id", "player_won"}
        ]
    if not feature_cols:
        raise ValueError("No predictors selected. Check extraction and the configured feature lists.")
    if {"battle_id", "player_won"}.intersection(feature_cols):
        raise ValueError("Identifiers and target labels cannot be predictors.")
    for name, frame in (("training", train_df), ("test", test_df)):
        missing = sorted(set(feature_cols) - set(frame.columns))
        if missing:
            message = f"Missing {name} features: {missing}"
            if USE_FEATURE_SELECTION:
                message += (
                    ". Feature selection requires every entry in ALL_FEATURES. "
                    "Enable the relevant feature extraction calls and rerun extraction, "
                    "or explicitly restrict ALL_FEATURES to the intended candidate pool."
                )
            raise ValueError(message)
    unknown_categorical = sorted(set(CATEGORICAL_FEATURES) - set(feature_cols))
    if unknown_categorical:
        raise ValueError(f"Categorical columns must be selected: {unknown_categorical}")

    X = train_df.loc[:, feature_cols].copy()
    X_test = test_df.loc[:, feature_cols].copy()
    y = train_df["player_won"].copy()
    if y.isna().any() or not y.isin([0, 1]).all():
        raise ValueError("player_won must contain only 0 and 1.")
    y = y.astype(int)

    categorical_features = [
        col for col in feature_cols
        if col in CATEGORICAL_FEATURES
        or not pd.api.types.is_numeric_dtype(X[col])
    ]
    for frame in (X, X_test):
        for col in feature_cols:
            if col in categorical_features:
                frame[col] = frame[col].astype("string").fillna("__unknown__").astype(str)
            else:
                # CatBoost handles missing numeric values natively.
                frame[col] = pd.to_numeric(frame[col], errors="raise").replace(
                    [np.inf, -np.inf], np.nan
                )

    print(f"Selected features: {len(feature_cols)}; categorical: {len(categorical_features)}")
    return X, y, X_test, categorical_features

### Performance metrics

In [ ]:
# --- METRICS ---
def compute_metrics(y_true: pd.Series | np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> dict:
    """
    Return accuracy, precision, recall, F1, and ROC-AUC for binary predictions.
    y_true and y_pred contain 0/1 labels.
    y_prob contains probabilities of class 1
    """

    return {
        # (TP + TN) / total
        "accuracy": accuracy_score(y_true, y_pred),
        # TP / (TP + FP)
        "precision": precision_score(y_true, y_pred, zero_division=0),
        # TP / (TP + FN)
        "recall": recall_score(y_true, y_pred, zero_division=0),
        # 2 * (precision * recall) / (precision + recall) = 2 * TP / (2 * TP + FP + FN)
        "f1": f1_score(y_true, y_pred, zero_division=0),
        # Area Under the Receiver Operating Characteristic Curve
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

### Training

In [ ]:
def select_feature_subset(X, y, categorical_features, random_state=RANDOM_STATE, scope="final"):
    """Batch backward elimination; all decisions use only the supplied training data."""
    if not 0 < SELECTION_DROP_FRACTION < 1:
        raise ValueError("SELECTION_DROP_FRACTION must be between 0 and 1.")
    if not 1 <= SELECTION_MIN_FEATURES <= X.shape[1]:
        raise ValueError("SELECTION_MIN_FEATURES must be between 1 and the candidate count.")
    if SELECTION_MAX_ITERATIONS < 1 or SELECTION_SHAP_SAMPLE_SIZE < 1:
        raise ValueError("Selection iterations and SHAP sample size must be positive.")
    if SELECTION_ACCURACY_TOLERANCE < 0:
        raise ValueError("SELECTION_ACCURACY_TOLERANCE cannot be negative.")

    # Reuse these three disjoint partitions at every elimination step.
    inner_train, tune_idx = train_test_split(
        np.arange(len(X)), test_size=SELECTION_VALIDATION_FRACTION,
        stratify=y, random_state=random_state,
    )
    fit_idx, stop_idx = train_test_split(
        inner_train, test_size=EARLY_STOPPING_FRACTION,
        stratify=y.iloc[inner_train], random_state=random_state,
    )
    shap_idx = np.random.default_rng(random_state).choice(
        fit_idx, size=min(SELECTION_SHAP_SAMPLE_SIZE, len(fit_idx)), replace=False,
    )
    active = X.columns.tolist()
    variable = X.iloc[fit_idx].nunique(dropna=False).gt(1)
    if not variable.any():
        raise ValueError("All candidate features are constant in the inner fitting data.")
    records = []
    while True:
        cats = [col for col in categorical_features if col in active]
        model = build_model(
            random_state=random_state,
            iterations=min(SELECTION_MAX_ITERATIONS, CATBOOST_PARAMS["iterations"]),
        )
        model.fit(
            X.iloc[fit_idx].loc[:, active], y.iloc[fit_idx], cat_features=cats,
            eval_set=(X.iloc[stop_idx].loc[:, active], y.iloc[stop_idx]),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS, use_best_model=True,
        )
        probability = model.predict_proba(X.iloc[tune_idx].loc[:, active])[:, 1]
        accuracy = accuracy_score(y.iloc[tune_idx], probability >= DECISION_THRESHOLD)
        records.append({
            "scope": scope,
            "feature_count": len(active),
            "inner_tuning_accuracy": accuracy,
            "iterations": model.tree_count_,
            "features": json.dumps(active),
        })
        print(f"  {scope}: {len(active)} features | inner tuning accuracy: {accuracy:.4f}")
        if len(active) == SELECTION_MIN_FEATURES:
            break

        # Recompute SHAP after each batch removal so redundant features can take over.
        shap_values = model.get_feature_importance(
            Pool(X.iloc[shap_idx].loc[:, active], cat_features=cats),
            type="ShapValues", shap_calc_type="Approximate",
        )
        importance = np.abs(shap_values[:, :-1]).mean(axis=0)
        # Zero SHAP ties must not leave a constant-only subset.
        importance[~variable.loc[active].to_numpy()] = -np.inf
        keep_count = max(
            SELECTION_MIN_FEATURES,
            int(len(active) * (1 - SELECTION_DROP_FRACTION)),
        )
        keep = {active[i] for i in np.argsort(-importance, kind="stable")[:keep_count]}
        active = [col for col in active if col in keep]

    history = pd.DataFrame(records)
    best_score = history["inner_tuning_accuracy"].max()
    eligible = history[
        history["inner_tuning_accuracy"] >= best_score - SELECTION_ACCURACY_TOLERANCE
    ]
    chosen = eligible["feature_count"].idxmin()
    history["selected"] = history.index == chosen
    selected = json.loads(history.loc[chosen, "features"])
    print(f"  {scope}: selected {len(selected)} features (best inner score: {best_score:.4f}).")
    return selected, history


def train_and_evaluate_cv(
    X: pd.DataFrame, y: pd.Series, categorical_features: list,
    k: int = K_FOLDS, n_repeats: int = N_REPEATS,
    random_state: int = RANDOM_STATE, use_feature_selection: bool = False,
) -> tuple:
    """
    Evaluate the complete selection/training procedure on untouched outer folds
    """

    splitter = RepeatedStratifiedKFold(
        n_splits=k, n_repeats=n_repeats, random_state=random_state
    )
    oof_probs = np.zeros(len(X))
    oof_counts = np.zeros(len(X), dtype=int)
    importances = pd.Series(0.0, index=X.columns)
    records = []
    selection_histories = []
    print(f"Starting {n_repeats} x {k}-fold CatBoost validation.")

    for i, (train_idx, val_idx) in enumerate(splitter.split(X, y)):
        repeat, fold = divmod(i, k)
        seed = random_state + i
        selected = X.columns.tolist()
        if use_feature_selection:
            selected, history = select_feature_subset(
                X.iloc[train_idx], y.iloc[train_idx], categorical_features,
                random_state=seed, scope=f"repeat_{repeat + 1}_fold_{fold + 1}",
            )
            selection_histories.append(history)
        fold_X = X.loc[:, selected]
        cats = [col for col in categorical_features if col in selected]
        fit_idx, stop_idx = train_test_split(
            train_idx,
            test_size=EARLY_STOPPING_FRACTION,
            stratify=y.iloc[train_idx],
            random_state=seed,
        )

        # The scored validation fold never participates in early stopping.
        stopping_model = build_model(random_state=seed)
        stopping_model.fit(
            fold_X.iloc[fit_idx], y.iloc[fit_idx],
            cat_features=cats,
            eval_set=(fold_X.iloc[stop_idx], y.iloc[stop_idx]),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            use_best_model=True,
        )
        iterations = max(1, stopping_model.get_best_iteration() + 1)
        model = build_model(random_state=seed, iterations=iterations)
        model.fit(fold_X.iloc[train_idx], y.iloc[train_idx], cat_features=cats)

        train_prob = model.predict_proba(fold_X.iloc[train_idx])[:, 1]
        val_prob = model.predict_proba(fold_X.iloc[val_idx])[:, 1]
        train_metrics = compute_metrics(
            y.iloc[train_idx], (train_prob >= DECISION_THRESHOLD).astype(int), train_prob
        )
        val_metrics = compute_metrics(
            y.iloc[val_idx], (val_prob >= DECISION_THRESHOLD).astype(int), val_prob
        )
        oof_probs[val_idx] += val_prob
        oof_counts[val_idx] += 1
        importances.loc[selected] += model.feature_importances_
        records.append({
            "repeat": repeat + 1,
            "fold": fold + 1,
            "iterations": iterations,
            "feature_count": len(selected),
            "selected_features": json.dumps(selected),
            **{f"train_{name}": value for name, value in train_metrics.items()},
            **{f"val_{name}": value for name, value in val_metrics.items()},
        })
        print(
            f"Repeat {repeat + 1}/{n_repeats} | Fold {fold + 1}/{k} | "
            f"Features: {len(selected)} | Trees: {iterations} | Train accuracy: {train_metrics['accuracy']:.4f} | "
            f"Validation accuracy: {val_metrics['accuracy']:.4f} | "
            f"ROC-AUC: {val_metrics['roc_auc']:.4f}"
        )

    if not np.all(oof_counts == n_repeats):
        raise RuntimeError("Every battle must receive one OOF prediction per repeat.")
    oof_probs /= oof_counts
    cv_results = pd.DataFrame(records)
    oof_metrics = compute_metrics(
        y, (oof_probs >= DECISION_THRESHOLD).astype(int), oof_probs
    )
    summary = pd.DataFrame({
        "train_mean": {name: cv_results[f"train_{name}"].mean() for name in oof_metrics},
        "validation_mean": {name: cv_results[f"val_{name}"].mean() for name in oof_metrics},
        "validation_std": {name: cv_results[f"val_{name}"].std(ddof=0) for name in oof_metrics},
        "oof": oof_metrics,
    })
    print("\nCatBoost validation summary (accuracy is the competition metric):")
    display(summary)
    feature_importance = pd.Series(
        importances / len(records), index=X.columns, name="mean_cv_importance"
    ).sort_values(ascending=False)
    selection_history = (
        pd.concat(selection_histories, ignore_index=True) if selection_histories else pd.DataFrame()
    )
    return cv_results, oof_probs, feature_importance, summary, selection_history


def export_run_report(
    summary: pd.DataFrame,
    used_features: list[str],
    use_allowed_features: bool,
    output_dir: str | Path = REPORT_DIR,
    selection_history: pd.DataFrame | None = None,
    cv_results: pd.DataFrame | None = None,
    candidate_features: list[str] | None = None,
) -> Path:
    """Save the displayed validation summary and the actual selected predictors."""
    created_at = datetime.now(timezone.utc)
    report_dir = Path(output_dir)
    report_dir.mkdir(parents=True, exist_ok=True)
    report_path = report_dir / f"catboost_report_{created_at:%Y%m%dT%H%M%S_%fZ}.md"

    lines = [
        "# CatBoost run report",
        "",
        f"Created (UTC): {created_at:%Y-%m-%d %H:%M:%S}",
        "",
        "## 1. CatBoost validation summary",
        "",
        "Accuracy is the competition metric. Validation std uses ddof=0.",
        "",
        "| Metric | " + " | ".join(summary.columns) + " |",
        "| --- | " + " | ".join("---:" for _ in summary.columns) + " |",
    ]
    for metric, row in summary.iterrows():
        lines.append(
            f"| {metric} | " + " | ".join(f"{value:.6f}" for value in row) + " |"
        )
    lines.extend([
        "",
        "## 2. Used features",
        "",
        f"ALLOWED_FEATURES filter: {'enabled' if use_allowed_features else 'disabled'}",
        f"Feature count: {len(used_features)}",
        "",
    ])
    lines.extend(f"- `{feature}`" for feature in used_features)
    if selection_history is not None and not selection_history.empty:
        prefix = report_path.with_suffix("")
        history_path = prefix.with_name(prefix.name + "_selection_history.csv")
        folds_path = prefix.with_name(prefix.name + "_folds.csv")
        stability_path = prefix.with_name(prefix.name + "_feature_stability.csv")
        selection_history.to_csv(history_path, index=False)
        cv_results.to_csv(folds_path, index=False)
        fold_features = cv_results["selected_features"].map(json.loads)
        stability = pd.DataFrame({
            "feature": candidate_features,
            "selected_folds": [
                sum(feature in subset for subset in fold_features)
                for feature in candidate_features
            ],
        })
        stability["selection_frequency"] = stability["selected_folds"] / len(cv_results)
        stability["in_final_subset"] = stability["feature"].isin(used_features)
        stability.sort_values(
            ["selection_frequency", "feature"], ascending=[False, True]
        ).to_csv(stability_path, index=False)
        unavailable = [col for col in dict.fromkeys(ALL_FEATURES) if col not in candidate_features]
        final_choice = selection_history[
            (selection_history["scope"] == "final") & selection_history["selected"]
        ].iloc[0]
        lines.extend([
            "", "## 3. Feature selection", "",
            "Method: SHAP-guided batch backward elimination, nested inside outer CV.",
            "Validation mean evaluates the selection procedure. Each outer fold may use a different subset.",
            "The final subset above is selected using all training labels; its inner score is a tuning score, not independent validation.",
            "This approximate path does not test every combination or guarantee the globally best subset.",
            "",
            f"Candidate features actually searched: {len(candidate_features)}",
            f"ALL_FEATURES not extracted: {len(unavailable)}",
            f"Fraction removed per step: {SELECTION_DROP_FRACTION}",
            f"Minimum features: {SELECTION_MIN_FEATURES}",
            f"Search tree limit: {SELECTION_MAX_ITERATIONS}",
            f"SHAP sample limit: {SELECTION_SHAP_SAMPLE_SIZE} (Approximate SHAP)",
            f"Inner tuning fraction: {SELECTION_VALIDATION_FRACTION}",
            f"Accuracy tolerance favoring smaller subsets: {SELECTION_ACCURACY_TOLERANCE}",
            f"Final subset inner tuning accuracy (optimistic): {final_choice['inner_tuning_accuracy']:.6f}",
            "",
            f"[All search steps]({history_path.name})",
            f"[Outer fold metrics and subsets]({folds_path.name})",
            f"[Selection frequency by feature]({stability_path.name})",
        ])
        if unavailable:
            lines.extend(["", "Unavailable ALL_FEATURES (extraction unchanged):", ""])
            lines.extend(f"- `{feature}`" for feature in unavailable)
    report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"Run report saved to '{report_path.resolve()}'.")
    return report_path


X_train, y_train, X_test, categorical_features = prepare_features(train_df, test_df)
candidate_features = X_train.columns.tolist()
cv_results, oof_probabilities, feature_importance, validation_summary, selection_history = train_and_evaluate_cv(
    X_train, y_train, categorical_features, use_feature_selection=USE_FEATURE_SELECTION,
)

selected_features = candidate_features
if USE_FEATURE_SELECTION:
    # The final subset is chosen without using the outer validation scores.
    selected_features, final_history = select_feature_subset(
        X_train, y_train, categorical_features, scope="final",
    )
    selection_history = pd.concat([selection_history, final_history], ignore_index=True)
    X_train = X_train.loc[:, selected_features]
    X_test = X_test.loc[:, selected_features]
    categorical_features = [col for col in categorical_features if col in selected_features]
    print(f"\nFinal selected features ({len(selected_features)}): {selected_features}")

final_iterations = max(1, int(np.median(cv_results["iterations"])))
print(f"\nFitting one final CatBoost model on {len(X_train)} battles with {final_iterations} trees.")
final_model = build_model(iterations=final_iterations)
final_model.fit(X_train, y_train, cat_features=categorical_features)
test_probabilities = final_model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= DECISION_THRESHOLD).astype(int)

report_path = export_run_report(
    validation_summary, selected_features,
    use_allowed_features=USE_ALLOWED_FEATURES and not USE_FEATURE_SELECTION,
    selection_history=selection_history, cv_results=cv_results,
    candidate_features=candidate_features,
)

## 4. Submission File

In [ ]:
def predict_and_save(predictions: np.ndarray, test_battle_ids: pd.Series, output_path: str = SUBMISSION_PATH, sample_path: str = SAMPLE_SUBMISSION_PATH) -> None:
    """
    Format and save the submission file using the required template
    """
    
    print(f"Saving predictions to '{output_path}'...")
    sample_submission = pd.read_csv(sample_path)
    expected_columns = ["battle_id", "player_won"]
    if list(sample_submission.columns) != expected_columns:
        raise ValueError(f"ERROR: expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

    predictions_by_id = pd.Series(predictions, index=test_battle_ids)
    sample_submission["player_won"] = sample_submission["battle_id"].map(predictions_by_id)

    if sample_submission["player_won"].isna().any():
        raise ValueError(f"ERROR: some battle IDs in the '{sample_path}' have no matching prediction.")

    sample_submission["player_won"] = sample_submission["player_won"].astype(int)
    sample_submission.to_csv(output_path, index=False)

    print(f"SUCCESS: file '{output_path}' created.")
    print(sample_submission.head(10))
    print(f"Prediction distribution (0 vs 1): {dict(sample_submission['player_won'].value_counts())}")


# Submission file
predict_and_save(test_predictions, test_df["battle_id"], SUBMISSION_PATH, SAMPLE_SUBMISSION_PATH)

## Debug

In [ ]:
display(feature_importance.head(25).to_frame())

if DEBUG:
    display(cv_results)
    display(X_train.describe(include="all"))